In [1]:
import pandas as pd
import numpy as np
import os

df = pd.read_csv('../data/classified/ola_reviews_classified.csv')
df['at'] = pd.to_datetime(df['at'])
monthly = pd.read_csv('../data/cleaned/ola_monthly_trends.csv')

print(f"Dataset loaded: {df.shape}")

Dataset loaded: (7119, 36)


In [2]:
# All numbers used in the report — computed live from data
# Never hardcoded — always derived so report stays accurate

total_reviews     = len(df)
date_start        = df['at'].min().strftime('%B %Y')
date_end          = df['at'].max().strftime('%B %Y')
overall_avg_rating = df['score'].mean()
pct_one_star      = (df['score'] == 1).mean() * 100
pct_negative_sent = (df['sentiment_label'] == 'Negative').mean() * 100
reply_rate        = df['developer_replied'].mean() * 100

# Category stats
cat_cols = [
    'cat_service_center', 'cat_software_app', 'cat_battery_range',
    'cat_customer_care', 'cat_spare_parts', 'cat_warranty_refunds',
    'cat_delivery_registration', 'cat_safety_breakdown', 'cat_pricing_value'
]

label_map = {
    'cat_service_center'       : 'Service Center',
    'cat_software_app'         : 'Software / App',
    'cat_battery_range'        : 'Battery & Range',
    'cat_customer_care'        : 'Customer Care',
    'cat_spare_parts'          : 'Spare Parts',
    'cat_warranty_refunds'     : 'Warranty & Refunds',
    'cat_delivery_registration': 'Delivery & Registration',
    'cat_safety_breakdown'     : 'Safety & Breakdown',
    'cat_pricing_value'        : 'Pricing & Value'
}

cat_stats = {}
for col in cat_cols:
    subset = df[df[col] == True]
    cat_stats[label_map[col]] = {
        'count'         : len(subset),
        'pct'           : len(subset) / total_reviews * 100,
        'avg_sentiment' : subset['sentiment_score'].mean(),
        'avg_rating'    : subset['score'].mean(),
        'pct_one_star'  : (subset['score'] == 1).mean() * 100
    }

# Multi-label stats
multi_2plus = (df['complaint_count'] >= 2).sum()
multi_3plus = (df['complaint_count'] >= 3).sum()
pct_multi   = multi_2plus / total_reviews * 100

# Pre vs post crisis
pre_mean  = df[df['at'] < '2025-07-01']['sentiment_score'].mean()
post_mean = df[df['at'] >= '2025-07-01']['sentiment_score'].mean()

print("Summary statistics computed.")
print(f"\nTotal reviews   : {total_reviews:,}")
print(f"Date range      : {date_start} → {date_end}")
print(f"Avg rating      : {overall_avg_rating:.2f}")
print(f"% one-star      : {pct_one_star:.1f}%")
print(f"Reply rate      : {reply_rate:.1f}%")
print(f"Multi-complaint : {pct_multi:.1f}% of reviews")

Summary statistics computed.

Total reviews   : 7,119
Date range      : May 2022 → June 2026
Avg rating      : 2.27
% one-star      : 59.9%
Reply rate      : 0.0%
Multi-complaint : 25.5% of reviews


In [4]:
report = f"""# OLA Electric — Voice of Customer Crisis Analysis
## Findings Summary & Business Recommendations

**Prepared by:** Naren Karthikeyan A  
**Data period:** {date_start} — {date_end}  
**Total reviews analyzed:** {total_reviews:,}  
**Methodology:** Multi-label keyword classification + VADER sentiment analysis  
**Validation:** 99.2% classifier agreement rate on 248 manually labeled reviews

---

## 1. Executive Summary

OLA Electric's customer experience crisis is real, measurable, and sustained.
Analysis of {total_reviews:,} Google Play Store reviews spanning four years
reveals an overall average rating of {overall_avg_rating:.2f}/5.0, with
{pct_one_star:.1f}% of reviews rated 1-star. The crisis is not driven by a
single failure — it is a systemic breakdown across software, after-sales
service, and customer support, with each failure amplifying the others.
Despite a partial sentiment recovery in 2026, average ratings remain below
3.0 and the brand has not returned to positive sentiment territory at any
point since early 2023.

---

## 2. Key Findings

### Finding 1 — Software is the Most Widespread Failure
**Observation:** Software/App complaints appear in
{cat_stats['Software / App']['pct']:.1f}% of all reviews
({cat_stats['Software / App']['count']:,} reviews) — the dominant complaint
category by a significant margin, nearly 5x the next highest category.

**Evidence:** Multi-label keyword classifier, 99.2% validated accuracy.
Average star rating for Software/App complaints: 
{cat_stats['Software / App']['avg_rating']:.2f}/5.0.
{cat_stats['Software / App']['pct_one_star']:.1f}% of these reviews are 1-star.

**Implication:** The OLA Electric app is the primary interface between the
customer and their scooter. Every app failure degrades the entire ownership
experience regardless of whether the physical product is functioning correctly.

---

### Finding 2 — Customer Care is the Most Severe Failure
**Observation:** Customer Care complaints produce a {cat_stats['Customer Care']['pct_one_star']:.1f}%
one-star rate — the highest of any complaint category — and the lowest average
rating ({cat_stats['Customer Care']['avg_rating']:.2f}/5.0).

**Evidence:** Block 7 rating heatmap. Kruskal-Wallis confirms sentiment
differs significantly across categories (H=274.324, p<0.001).

**Implication:** When a customer contacts OLA's support system and receives
no resolution, the relationship is effectively lost. Customer care failure
is the single most consistent predictor of the worst possible customer outcome.

---

### Finding 3 — Service Center and Customer Care Failures Are Structurally Linked
**Observation:** Service Center and Customer Care complaints co-occur in
271 reviews — the 5th highest co-occurrence pair. Service Center carries
{cat_stats['Service Center']['pct_one_star']:.1f}% one-star rate and
{cat_stats['Service Center']['avg_rating']:.2f} average rating.

**Evidence:** Co-occurrence analysis, Block 6. Service center delays generate
customer care contact; customer care non-response amplifies service frustration.
These are not independent failures.

**Implication:** Fixing one without the other produces incomplete relief.
A customer whose scooter is repaired but whose 47 calls went unanswered
will still write a 1-star review.

---

### Finding 4 — App Failure Connects Every Other Complaint
**Observation:** Software/App appears in 6 of the top 10 co-occurring
complaint pairs, including the dominant pair:
Software/App + Battery & Range (558 co-occurrences).

**Evidence:** Co-occurrence heatmap and top 10 pairs table, Block 6.

**Implication:** The app is not just a standalone complaint — it is the
connective tissue through which every other failure is experienced. Battery
status is read through the app. Service is booked through the app. Complaints
are raised through the app. When the app fails, every other system becomes
inaccessible simultaneously.

---

### Finding 5 — A Satisfied Customer Segment Exists and Is Recoverable
**Observation:** {(df['cat_positive_experience']==True).sum():,} reviews
({(df['cat_positive_experience']==True).mean()*100:.1f}%) contain explicit
positive language. 25 reviews simultaneously flag Software/App complaint
AND Positive Experience — customers who like the scooter but hate the app.

**Evidence:** Block 9 positive experience contrast analysis.

**Implication:** OLA Electric has a core of satisfied customers who value
the physical product. These customers are at risk of churning due to software
and service failures rather than product dissatisfaction. They are the
highest-priority retention segment.

---

### Finding 6 — The Crisis Has a Measurable Arc
**Observation:** Monthly average sentiment declined from +0.38 in mid-2022
to a crisis floor of -0.32 in early 2025, followed by partial recovery
to near-zero in 2026. Review volume peaked at 667 in July 2025 — 274%
above the previous month.

**Evidence:** Block 4 temporal trend analysis. Mann-Whitney U confirms
pre vs post-July 2025 sentiment shift is statistically significant
(p=3.74e-50).

**Implication:** The crisis had a beginning and shows directional improvement.
However, improvement in sentiment scores reflects reviewer population change
(disengagement of most negative customers) rather than genuine service
recovery — average star ratings remain below 3.0 throughout 2026.

---

### Finding 7 — Zero Developer Engagement Across Four Years
**Observation:** OLA Electric has maintained a 0% developer response rate
on Google Play Store across {total_reviews:,} reviews spanning four years.

**Evidence:** replyContent column — 100% null across entire dataset.

**Implication:** Industry benchmark for consumer apps is 40-60% response
rate on negative reviews. Zero engagement on a public platform signals
to prospective customers that complaints are ignored. A single visible,
empathetic response to a high-thumbs-up complaint costs minutes and
communicates accountability to thousands of readers.

---

### Finding 8 — Longer Reviews Signal Deeper Damage
**Observation:** Word count is negatively correlated with sentiment score
(Spearman r=-0.369, p<0.001) — the strongest correlation in this project.
Longer reviews are almost always more negative.

**Evidence:** Test 4, Section B statistical validation.

**Implication:** Review length is a reliable early-warning signal for
high-severity complaints. A complaint monitoring system that prioritizes
long, negative reviews for immediate human escalation would surface the
most damaged customer relationships efficiently.

---

## 3. Root Cause Analysis

The data supports a single unified root cause narrative:

> OLA Electric scaled customer acquisition faster than it scaled
> customer experience infrastructure.

The evidence chain:

Aggressive sales growth (2022-2024)

↓

Service center capacity not scaled proportionally

↓

Repair backlogs build → spare parts shortages emerge

↓

Customers unable to get resolutions → flood customer care

↓

Customer care overwhelmed → tickets expire unresolved

↓

App fails to provide visibility or alternative resolution path

↓

Customers escalate to public platforms (Play Store, Twitter)

↓

Rating decline → new buyer hesitation → market share loss

↓

38.83% market share (Jul 2024) → 17.35% (Jul 2025)

Every complaint category in this analysis is a symptom of this
single structural failure. The categories are not independent problems
— they are linked consequences of scaling sales without scaling service.

---

## 4. Business Recommendations

Prioritized by: severity (avg rating + one-star rate) × volume × fixability

---

### Recommendation 1 — Fix the App Immediately (High Volume, High Fixability)
**Priority:** Urgent  
**Category addressed:** Software / App (50.5% of reviews)  
**Function owner:** Product & Engineering  

The app is broken for a significant proportion of users — login failures,
connectivity errors, incorrect data display, and missing features are
documented across thousands of reviews. Unlike service center capacity
(which requires physical infrastructure investment), app fixes are
deployable immediately through OTA updates.

**Specific actions:**
- Prioritize connectivity stability (Bluetooth F002 error appears repeatedly)
- Fix login/authentication failures (blank screen, loading loops)
- Restore accurate battery and location display
- Implement push notifications for charging status
- Add dark mode and ride history (repeatedly requested features)

**Expected impact:** Reducing Software/App complaint rate by 50% would
remove this complaint from ~25% of all reviews — the single largest
improvement available through any intervention.

---

### Recommendation 2 — Rebuild Customer Care as Priority Infrastructure
**Priority:** Critical  
**Category addressed:** Customer Care (10.0% of reviews, 93.9% one-star rate)  
**Function owner:** Customer Experience  

Customer care failure is the most reliable predictor of the worst customer
outcome in this dataset. A 93.9% one-star rate means almost no customer
who contacts support walks away satisfied. This is not a training problem
— it is a structural capacity and process problem.

**Specific actions:**
- Publish and enforce maximum response time SLAs (target: 24 hours)
- Implement ticket tracking visible to customers in the app
- Escalation path: unresolved tickets auto-escalate to senior staff at 48h
- Begin responding to Google Play Store reviews — start with
  high-thumbs-up negative reviews (publicly visible, high reach)
- Report monthly resolution rate as a public metric (accountability signal)

**Expected impact:** Even a 30% improvement in customer care resolution
rate would directly address the category with the highest damage-per-customer
ratio in the dataset.

---

### Recommendation 3 — Treat Service Center + Spare Parts as One System
**Priority:** High  
**Categories addressed:** Service Center (10.4%), Spare Parts (2.0%)  
**Function owner:** Operations + Procurement  

Co-occurrence analysis confirms these failures travel together.
A customer whose scooter sits at a service center for months waiting
for parts experiences both failures simultaneously. Spare Parts has
the most negative average sentiment (-0.385) despite the lowest volume
— when parts are unavailable, the experience is catastrophic.

**Specific actions:**
- Implement minimum spare parts inventory requirements at each service center
- Create a parts availability dashboard visible to service center staff
- Publish estimated repair timelines at job card creation (set expectations)
- Proactive SMS/app notification when parts arrive and repair begins
- Target: maximum 15-day repair turnaround for common components

**Expected impact:** Reducing service center wait times addresses the
category with the second-lowest average rating (1.28) and highest
co-occurrence with customer care escalation.

---

### Recommendation 4 — Activate Play Store as a Customer Recovery Channel
**Priority:** Medium  
**Finding addressed:** 0% developer response rate across 4 years  
**Function owner:** Customer Experience / Marketing  

Zero responses across {total_reviews:,} reviews is a missed recovery
opportunity at scale. Each response to a public review is read by
hundreds of prospective customers evaluating OLA Electric.

**Specific actions:**
- Assign dedicated resource for Play Store review responses
- Priority queue: 1-star reviews with 10+ thumbs-up (highest reach)
- Response template: acknowledge + ticket number + resolution timeline
- Target: 40% response rate on 1-star reviews within 30 days
- Track: does responded-to review get updated to higher rating?

**Expected impact:** Measurable improvement in prospective buyer perception.
Visible accountability signals to the market that OLA Electric takes
complaints seriously — partially offsetting negative rating impact.

---

### Recommendation 5 — Implement Early Warning System
**Priority:** Medium-term  
**Finding addressed:** Review length correlates with damage severity  
**Function owner:** Data / Product  

Word count is the strongest predictor of sentiment severity in this
dataset (Spearman r=-0.369). A simple automated system monitoring
review length + sentiment score could surface high-damage complaints
for immediate human escalation before they accumulate thumbs-up votes
and influence prospective buyers.

**Specific actions:**
- Build automated Play Store review monitoring pipeline
- Flag: review length > 100 words AND sentiment score < -0.5
- Auto-create internal ticket for flagged reviews within 2 hours
- Monthly dashboard: complaint category trend with 3-month forecast

---

## 5. KPIs to Track Recovery

If OLA Electric implements these recommendations, the following
metrics — all measurable from public Play Store data — would indicate
genuine recovery:

| KPI | Current | Target | Timeframe |
|-----|---------|--------|-----------|
| Overall avg star rating | {overall_avg_rating:.2f} | 3.5+ | 12 months |
| % 1-star reviews | {pct_one_star:.1f}% | <35% | 12 months |
| Software/App complaint rate | 50.5% | <25% | 6 months |
| Customer Care 1-star rate | 93.9% | <70% | 9 months |
| Developer response rate | 0% | 40%+ | 3 months |
| Avg sentiment score | {df['sentiment_score'].mean():.3f} | >0.10 | 12 months |
| Multi-complaint review rate | {pct_multi:.1f}% | <15% | 12 months |

These KPIs can be tracked monthly using the same scraping and
classification pipeline built in this project — making this analysis
a living monitoring tool, not a one-time report.

---

## 6. Methodological Notes

| Decision | Rationale |
|----------|-----------|
| VADER over BERT | Lexicon-based, explainable, validated at 82.2% positive agreement. Upgrade not warranted by evidence. |
| Rule-based over LDA | Quantifying known problems not discovering unknown ones. 99.2% manual validation agreement. |
| Multi-label classification | Real complaints span multiple categories. Single-label would discard signal. |
| Play Store as primary source | Public, free, no authentication, 4-year temporal coverage, Python-native scraping. |
| Star rating as primary health metric | More reliable than VADER for absolute negative sentiment due to factual complaint language patterns in Indian English. |

---

*Analysis conducted using Python (pandas, VADER, scipy, matplotlib, seaborn).  
Full code and data available at: github.com/coderfox-cap*
"""

# Save
os.makedirs('../outputs/', exist_ok=True)
with open('../outputs/findings_summary.md', 'w', encoding='utf-8') as f:
    f.write(report)

print(f"✓ findings_summary.md saved.")
print(f"  Length: {len(report):,} characters")
print(f"  Lines : {len(report.splitlines()):,}")

✓ findings_summary.md saved.
  Length: 13,816 characters
  Lines : 353


In [5]:
verbal_summary = """
INTERVIEW VERBAL SUMMARY — OLA Electric VoC Analysis
=====================================================

OPENING (30 seconds):
"I scraped 7,119 Google Play Store reviews for OLA Electric
spanning four years, applied VADER sentiment analysis and a
validated multi-label keyword classifier to categorize complaints
across 10 business areas, and used statistical testing to confirm
which findings are significant versus coincidental."

THE THREE KEY FINDINGS (60 seconds):
"Three things stood out.

First — Software/App complaints dominate at 50.5% of all reviews,
but they're actually the least severe complaint by average rating.
The app is broken for a lot of people, but it's fixable.

Second — Customer Care is the real crisis. 93.9% of reviews
mentioning customer care are 1-star. When customers can't get help,
the relationship ends permanently. Kruskal-Wallis confirmed that
sentiment differs significantly across categories at p<0.001.

Third — OLA Electric has never responded to a single Play Store
review in four years. Across 7,119 reviews. That's not an oversight
— it's a systematic disengagement from their most public feedback
channel."

THE RECOMMENDATION (20 seconds):
"My top recommendation isn't to fix the most complained-about thing.
It's to fix the most damaging thing — customer care — while
simultaneously patching the app, which is a quick win at high volume.
Those two interventions together address the highest severity and
highest frequency failures simultaneously."

CLOSING (10 seconds):
"The same pipeline I built can monitor these metrics monthly going
forward. The project isn't just an analysis — it's a monitoring tool."
"""

print(verbal_summary)

with open('../outputs/interview_verbal_summary.txt', 'w') as f:
    f.write(verbal_summary)

print("✓ Interview summary saved.")


INTERVIEW VERBAL SUMMARY — OLA Electric VoC Analysis

OPENING (30 seconds):
"I scraped 7,119 Google Play Store reviews for OLA Electric
spanning four years, applied VADER sentiment analysis and a
validated multi-label keyword classifier to categorize complaints
across 10 business areas, and used statistical testing to confirm
which findings are significant versus coincidental."

THE THREE KEY FINDINGS (60 seconds):
"Three things stood out.

First — Software/App complaints dominate at 50.5% of all reviews,
but they're actually the least severe complaint by average rating.
The app is broken for a lot of people, but it's fixable.

Second — Customer Care is the real crisis. 93.9% of reviews
mentioning customer care are 1-star. When customers can't get help,
the relationship ends permanently. Kruskal-Wallis confirmed that
sentiment differs significantly across categories at p<0.001.

Third — OLA Electric has never responded to a single Play Store
review in four years. Across 7,119 reviews.